# Combinación de archivos SIVIGILA — Dengue grave (COD_EVE 220)

Lee los archivos `Datos_YYYY_220.xlsx` (2007–2024, **excluye 2025**) y los consolida en un único CSV con una columna `source_file` que identifica el año de origen.

**Salida:** `data/processed/sivigila_dengue_grave_consolidado.csv`

> **Nota:** Este dataset es independiente del dengue clásico (210). Verificar posibles duplicados por CONSECUTIVE antes de combinar ambos en análisis conjuntos.

In [ ]:
import os, glob, re, time
import pandas as pd

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR   = "../data/raw/grave"    # carpeta con los Datos_YYYY_220.xlsx
OUTPUT_CSV = "../data/processed/sivigila_dengue_grave_consolidado.csv"

EXCLUDE_YEARS = [2025]              # 2025 excluido: fuera del período de análisis

COLUMNS = None
# COLUMNS = [
#     "CONSECUTIVE", "COD_EVE", "FEC_NOT", "SEMANA", "ANO",
#     "COD_DPTO_O", "COD_MUN_O", "Departamento_ocurrencia", "Municipio_ocurrencia",
#     "TIP_CAS", "confirmados", "PAC_HOS", "CON_FIN", "EDAD", "UNI_MED", "SEXO",
# ]

In [ ]:
data_dir = os.path.abspath(DATA_DIR)
output   = os.path.abspath(OUTPUT_CSV)
os.makedirs(os.path.dirname(output), exist_ok=True)

files = sorted(glob.glob(os.path.join(data_dir, "Datos_*_220.xlsx")))
files = [f for f in files
         if int(re.search(r"\d{4}", os.path.basename(f)).group()) not in EXCLUDE_YEARS]

print(f"Archivos a procesar: {len(files)}  (excluidos: {EXCLUDE_YEARS})")
for f in files:
    mb = os.path.getsize(f) / 1_048_576
    print(f"  {os.path.basename(f)}  ({mb:.1f} MB)")

In [ ]:
total_rows  = 0
first_write = True
resumen     = []
t0          = time.time()

for path in files:
    fname = os.path.basename(path)
    year  = int(re.search(r"\d{4}", fname).group())
    mb    = os.path.getsize(path) / 1_048_576

    print(f"Leyendo {fname}  ({mb:.1f} MB)...", end="", flush=True)
    t1 = time.time()

    try:
        df = pd.read_excel(path, engine="openpyxl", usecols=COLUMNS, dtype=str)
    except Exception as e:
        print(f" ERROR: {e}")
        continue

    df.insert(0, "source_file", year)
    rows = len(df)
    total_rows += rows
    resumen.append({"año": year, "filas": rows})

    df.to_csv(output, mode="w" if first_write else "a",
              header=first_write, index=False, encoding="utf-8")
    first_write = False

    print(f" {rows:,} filas  ({time.time()-t1:.0f}s)")

print(f"\nTotal: {total_rows:,} filas  |  Tiempo: {(time.time()-t0)/60:.1f} min")
print(f"CSV guardado en: {output}")

In [ ]:
# Resumen por año
df_resumen = pd.DataFrame(resumen)
df_resumen["% del total"] = (df_resumen["filas"] / df_resumen["filas"].sum() * 100).round(1)
display(df_resumen.style.format({"filas": "{:,}", "% del total": "{:.1f}%"}))

In [ ]:
# Verificación rápida del CSV generado
df_check = pd.read_csv(output, nrows=5)
print(f"Columnas ({len(df_check.columns)}): {list(df_check.columns)[:10]} ...")
display(df_check.head())

In [ ]:
# Distribución de TIP_CAS — todos deberían ser 3 (grave) o similares
df_sample = pd.read_csv(output, usecols=["TIP_CAS", "source_file"])
print("Distribución TIP_CAS:")
display(df_sample["TIP_CAS"].value_counts().to_frame())